# 07 — KPI Optimization

Business decision simulation using LTV, retention, CAC and budget allocation.

**Project:** Marketing Analytics Causal & LTV Lab  
**Style:** Hands-on, advanced, interview-ready notebook  
**How to use:** Run cell by cell, inspect outputs, then discuss interpretation and pitfalls.


## Main notebook code

Run this notebook and then we will discuss the output, assumptions and pitfalls.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
PROCESSED_DIR=Path('../data/processed'); REPORTS_DIR=Path('../reports'); REPORTS_DIR.mkdir(parents=True, exist_ok=True)
wallet_path=PROCESSED_DIR/'wallet_ltv_predictions.csv'
customers=pd.read_csv(wallet_path) if wallet_path.exists() else None
kpi_tree={'Revenue':['Active Customers','ARPU'],'Active Customers':['New Customers','Retained Customers','Reactivated Customers'],'New Customers':['Marketing Spend','CAC','Conversion Rate'],'Retained Customers':['Churn Rate','Retention Campaign Reach','Retention Lift'],'ARPU':['Purchase Frequency','AOV','Discount Rate']}
for k,v in kpi_tree.items(): print(k,'->',', '.join(v))
if customers is not None:
    df=customers.copy(); df['predicted_ltv']=df.get('predicted_ltv',df['LTV']); df['churn_risk_proxy']=(df.Last_Transaction_Days_Ago-df.Last_Transaction_Days_Ago.min())/(df.Last_Transaction_Days_Ago.max()-df.Last_Transaction_Days_Ago.min()); df['target_score']=df.predicted_ltv*df.churn_risk_proxy; df['target_decile']=pd.qcut(df.target_score.rank(method='first'),10,labels=False)+1
    rows=[]
    for lift in [.02,.05,.08,.12]:
        for min_decile in [10,9,8,7]:
            targeted=df[df.target_decile>=min_decile]; profit=(targeted.predicted_ltv*lift-5).sum(); rows.append({'retention_lift':lift,'target_decile_min':min_decile,'customers_targeted':len(targeted),'expected_profit':profit,'profit_per_customer':profit/len(targeted)})
    display(pd.DataFrame(rows).sort_values('expected_profit',ascending=False).head(15))
channels=pd.DataFrame({'channel':['search','social','display','email'],'current_budget':[100000,80000,40000,20000],'base_roi':[2.8,2.1,1.4,1.8],'saturation':[180000,140000,70000,35000]})
def response(b,roi,sat): return roi*b*(1-np.exp(-b/sat))
total=channels.current_budget.sum(); rows=[]
for search in range(40000,181000,20000):
 for social in range(40000,161000,20000):
  for display in range(0,101000,20000):
   email=total-search-social-display
   if email<0 or email>80000: continue
   budgets={'search':search,'social':social,'display':display,'email':email}; rev=sum(response(budgets[r.channel],r.base_roi,r.saturation) for _,r in channels.iterrows()); rows.append({**budgets,'expected_revenue':rev})
alloc=pd.DataFrame(rows).sort_values('expected_revenue',ascending=False); display(alloc.head(10))
(REPORTS_DIR/'07_kpi_optimization_recommendations.md').write_text('# KPI Optimization

Decision simulation for retention targeting and budget allocation.
')


## Discussion prompts

1. What assumption is strongest here?
2. Which pitfall would break the conclusion?
3. How would you explain this to a non-technical stakeholder?
